In [3]:
from pynq import Overlay, MMIO
import pynq.lib as lib
from pynq import allocate
import numpy as np

ol = Overlay("/root/jupyter_notebooks/rhd_spi_0807/test_ila_data_o/rhd_spi_ip_0806_wrapper.bit")

In [4]:
ol?

In [5]:
# RHD SPI IP 的基地址 
RHD_SPI_BASE = 0xA0000000
ADDRESS_RANGE = 0x10000

# 暫存器偏移
CONTROL_REG = 0x00  # bit[0]=start_pulse, bit[1]=stop_pulse, bit[7:4]=phase_select
PHA_SEL_REG = 0x04  # [3:0]
TX_DATA_REG = 0x08  # 16-bit data to transmit
RX_DATA_REG = 0x0C  # 16-bit received data
STATUS_REG  = 0x10  # bit[0]=busy, bit[1]=finish
# 初始化 MMIO
mmio = MMIO(RHD_SPI_BASE, ADDRESS_RANGE)

# 讀取狀態
status = mmio.read(STATUS_REG)
print(f"SPI Status: 0x{status:08x}")
print(f"Busy: {status & 0x1}")
print(f"Finish: {(status >> 1) & 0x1}")

SPI Status: 0x00000001
Busy: 1
Finish: 0


In [4]:
from pynq import MMIO
BASE   = 0xA0000000
RANGE  = 0x10000
USER_REG = 0x14                 # slv_reg5

mmio = MMIO(BASE, RANGE)

patterns = [0xCAFEBABE, 0x0BAD_F00D, 0x55AA55AA]

for p in patterns:
    mmio.write(USER_REG, p)
    r = mmio.read(USER_REG)
    print(f"Write 0x{p:08X}  → Read 0x{r:08X} "
          f"{'OK' if r==p else 'FAIL'}")



Write 0xCAFEBABE  → Read 0xCAFEBABE OK
Write 0x0BADF00D  → Read 0x0BADF00D OK
Write 0x55AA55AA  → Read 0x55AA55AA OK


In [ ]:
# AXI 地址
CTRL    = 0x00 #reg0
STATUS  = 0x10 #reg3
RX_DATA = 0x0C #reg4
PHASE_SEL  = 0x04 #reg1

# 1️⃣ 送 start_pulse：bit0 = 1
mmio.write(PHASE_SEL, 0x2)
mmio.write(CTRL, 0x1)

# 2️⃣ 等 BUSY 變 0（或 FINISH 變 1）
# while mmio.read(STATUS) & 0x1:        # BUSY 還在
#     pass

# 3️⃣ 此時 FINISH = 1，讀 RX_DATA
data = mmio.read(RX_DATA) & 0xFFFF
print(f"RX = 0x{data:04X}")

In [9]:
def read_once():
    mmio.write(CTRL, 0x1)          # start_pulse
#     while mmio.read(STATUS) & 1:   # 等 busy=0
#         pass
    return mmio.read(RX_DATA) & 0xFFFF

results = {}
for ph in range(16):               # 0‥15 全掃
    mmio.write(PHASE_SEL, ph)
    results[ph] = read_once()

for ph,val in results.items():
    print(f"phase {ph:2d}: 0x{val:04X}")

phase  0: 0xE800
phase  1: 0xE900
phase  2: 0xD400
phase  3: 0xD400
phase  4: 0xD400
phase  5: 0xD200
phase  6: 0xD200
phase  7: 0xD200
phase  8: 0xD200
phase  9: 0xD200
phase 10: 0xD200
phase 11: 0xD200
phase 12: 0xD200
phase 13: 0xD200
phase 14: 0xD200
phase 15: 0xD200


In [16]:
CTRL    = 0x00
STATUS  = 0x10
RX_DATA = 0x0C
PHASE_SEL  = 0x04

def read_once():
    mmio.write(CTRL, 0x1)          # start_pulse
#     while mmio.read(STATUS) & 1:   # 等 busy=0
#         pass
    return mmio.read(RX_DATA) & 0xFFFF

results = {}
mmio.write(PHASE_SEL, 0x02)
for i in range(16):               
    results[i] = read_once()
    
for i,val in results.items():
    print(f"data {i:2d}: 0x{val:04X}")

data  0: 0xD400
data  1: 0xD600
data  2: 0xD000
data  3: 0xD200
data  4: 0xD400
data  5: 0xD600
data  6: 0xD800
data  7: 0xD000
data  8: 0xD000
data  9: 0xD200
data 10: 0xD400
data 11: 0xD600
data 12: 0xD800
data 13: 0xD800
data 14: 0xD000
data 15: 0xD200
